# 08 Лаборатория — Матрица выбора стратегии в действии

Пять рыночных конфигураций. Для каждой мы читаем **направление x уровень IV x горизонт**, собираем
2-3 конструкции-кандидата, которые предлагает матрица, из подходящей образцовой цепочки, сравниваем
их через `analyzer.summarize` + `viz.plot_compare` и дорассуждаем до выбора в markdown.

Цепочки: `DEMO` (спот 100, IV ~25%), `LOWVOL` (спот 185, IV ~14%), `HIGHVOL` (спот 62, IV ~55%).
IV rank указан для каждой конфигурации (образцы — это одиночные срезы; считайте указанный ранг
заданным).

Работает офлайн, сверху вниз.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, viz, data

demo = data.load_sample_chain("DEMO")
lowvol = data.load_sample_chain("LOWVOL")
highvol = data.load_sample_chain("HIGHVOL")

def brief(pos, spot, vol):
    s = analyzer.summarize(pos, spot, vol)
    print(f"{s['label']:<34} net={s['net_premium']:+8.0f} "
          f"maxP={s['max_profit']:>8} maxL={s['max_loss']:>8} POP={s['probability_of_profit']:.2f}")

## Конфигурация 1 — Нейтрально, ВЫСОКАЯ IV, 30 дней (HIGHVOL, спот 62)

IV rank ~75. Направленного уклона нет, ожидаем, что бумага примерно останется в диапазоне. Высокая
IV → **продаём премию**. Ячейка матрицы «нейтрально / высокая IV»: айрон кондор, айрон бабочка,
короткий стрэнгл. Кандидаты ниже (30 DTE).

In [ ]:
ic = strategies.iron_condor((50,0.54),(55,1.37),(70,1.32),(72.5,0.87), expiry=30/365)
ib = strategies.iron_butterfly((55,1.37),(62.5,4.14),(62.5,3.85),(70,1.32), expiry=30/365)
strangle = strategies.short_strangle((55,1.37),(70,1.32), expiry=30/365)
for p in (ic, ib, strangle):
    brief(p, 62, 0.56)

In [ ]:
ax = viz.plot_compare([ic, ib, strangle])
ax.set_title("Конфигурация 1: кондор против бабочки против короткого стрэнгла (HIGHVOL)")

**Рассуждение.** У короткого стрэнгла наибольшие кредит и POP, но **неограниченный риск** и
крупный расход покупательной способности — вето по тай-брейкеру, если только счёт это не тянет и вы
не готовы к поставке с любой из сторон. Айрон бабочка приносит самый большой *ограниченный* кредит,
но у неё узкая палатка прибыли (ставка на пин). **Айрон кондор** — сбалансированный выбор:
ограниченный риск, широкая зона прибыли под взгляд «останется в диапазоне», короткие страйки в районе
16-дельты. **Выбор: айрон кондор**, управлять на 50% / 21 DTE.

## Конфигурация 2 — Бычий взгляд, НИЗКАЯ IV, 60 дней (LOWVOL, спот 185)

IV rank ~15. Умеренно бычий взгляд, терпеливый. Низкая IV → **покупаем премию / дебет**. Матрица,
ячейка «бычий взгляд / низкая IV»: длинный колл, бычий **колл**-спред (дебетовый), колл-диагональ /
PMCC. Кандидаты используют цепочку 45 DTE (прокси под горизонт) плюс ногу на 90 DTE для диагонали.

In [ ]:
long_c = strategies.long_call((185, 4.4), expiry=45/365)
bull_call = strategies.bull_call_spread((185, 4.4), (195, 0.95), expiry=45/365)
diag = strategies.diagonal_spread("call", short=(195, 0.95), long=(180, 9.45),
                                  short_expiry=45/365, long_expiry=90/365)
for p in (long_c, bull_call, diag):
    brief(p, 185, 0.15)

In [ ]:
ax = viz.plot_compare([long_c, bull_call, diag])
ax.set_title("Конфигурация 2: длинный колл против бычьего колл-спреда против колл-диагонали (LOWVOL)")

**Рассуждение.** У голого длинного колла наибольший потенциал роста, но он платит полную
тету и полную вегу (дешёвую, а потому приемлемую) — при низкой IV это нормально, если вам нужна
выпуклость. **Бычий колл-спред** ограничивает рост на 195, но примерно вдвое снижает дебет и
ограничивает риск — самое чистое выражение взгляда «умеренно бычий, терпеливый» с известным
максимальным убытком. Диагональ добавляет уклон в длинную вегу и позволяет роллировать короткую ногу.
**Выбор: бычий колл-спред** ради ограниченного риска; апгрейд до диагонали, если вы хотите продавать
ближний месяц раз за разом. При низкой IV мы чистые *покупатели* — правильная сторона сделки по
волатильности.

## Конфигурация 3 — Бычий взгляд, ВЫСОКАЯ IV, 30 дней, готовы владеть (HIGHVOL, спот 62)

IV rank ~75. Взгляд бычий, но IV высока → **продаём премию** на бычьей стороне. Матрица:
обеспеченный деньгами пут, бычий **пут**-спред (кредитный), джейд-лизард. Вы были бы рады владеть
HIGHVOL по 55.

In [ ]:
csp = strategies.cash_secured_put((55, 1.37), expiry=30/365)
bull_put = strategies.bull_put_spread((57.5, 2.06), (52.5, 0.88), expiry=30/365)
jade = strategies.jade_lizard((55, 1.37), (67.5, 1.94), (70, 1.32), expiry=30/365)
for p in (csp, bull_put, jade):
    brief(p, 62, 0.56)
print("кредит лизарда на акцию:", round(-jade.net_premium()/100, 2), "ширина колл-спреда:", 2.5)

**Рассуждение.** Все три продают дорогую IV. У обеспеченного деньгами пута самое крупное
обязательство вниз (риск фактически тянется до нуля), но акции по 55 вам *нужны*. Бычий пут-спред
ограничивает риск. **Джейд-лизард** добавляет колл-спред сверху: сравните кредит на акцию с шириной
колл-спреда 2.5 — если кредит >= ширины, **риска вверх нет**. Здесь кредит ~ (1.37+1.94-1.32)=1.99 >
2.5? Нет (1.99 < 2.5), значит, небольшой риск вверх остаётся — сузьте спред или доберите кредит.
**Выбор: джейд-лизард, если удаётся выполнить кредит >= ширины; иначе бычий пут-спред** ради чистого
ограниченного риска.

## Конфигурация 4 — Нейтральный ПИН, умеренная IV, 45 дней (DEMO, спот 100)

IV rank ~40 (средний). Ожидаем, что DEMO запинуется около 100. Нейтральная ячейка, IV не является
явно высокой или низкой: **календарь** на цели пина (длинная вега дальнего месяца) против дешёвой
**длинной бабочки** с ограниченным риском.

In [ ]:
cal = strategies.calendar_spread("call", 100, front_expiry=21/365, front_premium=2.66,
                                 back_expiry=45/365, back_premium=3.91)
fly = strategies.long_call_butterfly((95, 7.05), (100, 3.91), (105, 1.85), expiry=45/365)
for p in (cal, fly):
    brief(p, 100, 0.26)

Календарь — конструкция со смешанными экспирациями: его настоящая палатка требует `pnl_at`
на экспирации ближней ноги, поэтому сравниваем выплату бабочки на экспирации с кривой календаря на
экспирации ближней ноги, а не наивным наложением.

In [ ]:
spots = np.linspace(88, 112, 121)
from optionslab import payoff
fig, ax = plt.subplots()
ax.plot(spots, payoff.pnl_curve(fly, spots), label="длинная бабочка (экспирация)")
ax.plot(spots, payoff.pnl_curve(cal, spots, t_elapsed=21/365, vol=0.26), label="календарь (эксп. ближней ноги)")
ax.axhline(0, color="k", lw=.7); ax.legend(); ax.set_title("Конфигурация 4: бабочка против календаря вокруг пина")

**Рассуждение.** Обе конструкции зарабатывают, если DEMO запинуется на 100. **Длинная
бабочка** — чистая ставка на пин с ограниченным риском и без взгляда на вегу. **Календарь** добавляет
длинную вегу — лучше, если вы вдобавок считаете, что IV удержится или вырастет, и хуже, если
надвигается обвал волатильности. **Выбор: бабочка, если взгляда на волатильность нет; календарь,
если вы ждёте стабильной или растущей IV.** Тай-брейкер: календарь требует управления по переоценке
через модель (управлять на экспирации ближней ноги).

## Конфигурация 5 — Сжатая пружина под БОЛЬШОЕ движение, НИЗКАЯ IV, 45 дней (LOWVOL, спот 185)

IV rank ~15, направление неизвестно, вы ждёте пробоя. «Нейтрально, но ждём движения» + низкая IV →
**длинная волатильность**: длинный стрэддл/стрэнгл или бэкспред. Длинную волатильность гораздо легче
обосновать, когда она дёшева.

In [ ]:
straddle = strategies.long_straddle((185, 4.4), (185, 3.49), expiry=45/365)
strangle_l = strategies.long_strangle((180, 1.69), (190, 2.22), expiry=45/365)
backspr = strategies.call_backspread((185, 4.4), (195, 0.95), expiry=45/365, ratio=(1, 2))
for p in (straddle, strangle_l, backspr):
    brief(p, 185, 0.15)

In [ ]:
ax = viz.plot_compare([straddle, strangle_l, backspr])
ax.set_title("Конфигурация 5: длинный стрэддл против длинного стрэнгла против колл-бэкспреда (LOWVOL)")

**Рассуждение.** **Длинный стрэддл** зарабатывает на большом движении в любую сторону, но
стоит дороже всех и истекает тетой, если LOWVOL стоит на месте. **Длинный стрэнгл** дешевле (шире
точки безубыточности, нужно большее движение). **Колл-бэкспред** — это направленная ставка на длинную
волатильность (вверх), часто почти в ноль по премии, с ограниченной долиной; берите его, только если
у вас есть уклон вверх. **Выбор: длинный стрэнгл** как дешёвая двусторонняя ставка на пробой при
низкой IV; бэкспред — если направленный уклон всё же есть. Всем трём нужен *рост* IV — что верно для
низковолатильного режима.

## Эксперименты

1. Конфигурация 1: перенесите короткие страйки кондора примерно на 10-дельту (шире:
   47.5/52.5/72.5/75). Как размениваются кредит и POP? Перезапустите `brief`.
2. Конфигурация 2: замените бычий колл-спред на бычий *пут*-спред в кредит и сравните — почему при
   **низкой** IV правильным выбором остаётся дебетовая версия?
3. Конфигурация 3: почините джейд-лизард так, чтобы кредит >= ширины колл-спреда (сузьте до
   65/67.5). Убедитесь, что риск вверх исчез.
4. Конфигурация 5: поднимите плоскую `vol`, которую вы передаёте в `brief`/`summarize`, и
   понаблюдайте за изменением POP — сделкам с длинной волатильностью нужен рост воли после входа.
5. Возьмите любую конфигурацию и поменяйте горизонт (DTE). Как более короткая или более длинная
   экспирация меняет кандидата, которого вы бы выбрали?